In [1]:
import tensorflow as tf
import numpy as np
import math
import os

In [3]:
# @title
# 3) Load the TFLite model with Interpreter and allocate tensors
interpreter = tf.lite.Interpreter(model_path='cnn_s_quantized.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
tensor_details = interpreter.get_tensor_details()

print("Input details:", input_details)
print("Output details:", output_details)
print("Number of tensors:", len(tensor_details))


Input details: [{'name': 'input', 'index': 0, 'shape': array([  1, 490], dtype=int32), 'shape_signature': array([ -1, 490], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (1.07421875, 102), 'quantization_parameters': {'scales': array([1.0742188], dtype=float32), 'zero_points': array([102], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'Identity', 'index': 20, 'shape': array([ 1, 12], dtype=int32), 'shape_signature': array([-1, 12], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00390625, -128), 'quantization_parameters': {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Number of tensors: 21


In [4]:
# @title
# 4) List all tensors (index, name, shape, dtype)
print("\nTensors:")
for t in tensor_details:
    print(f"  idx={t['index']:3d} | name={t['name'][:40]:40s} | shape={t['shape']} | dtype={t['dtype']}")



Tensors:
  idx=  0 | name=input                                    | shape=[  1 490] | dtype=<class 'numpy.int8'>
  idx=  1 | name=model/tf.reshape/Reshape/shape           | shape=[4] | dtype=<class 'numpy.int32'>
  idx=  2 | name=model/flatten/Const                      | shape=[2] | dtype=<class 'numpy.int32'>
  idx=  3 | name=model/conv2d/Conv2D                      | shape=[28 10  4  1] | dtype=<class 'numpy.int8'>
  idx=  4 | name=model/batch_normalization/FusedBatchNorm | shape=[28] | dtype=<class 'numpy.int32'>
  idx=  5 | name=model/conv2d_1/Conv2D                    | shape=[30 10  4 28] | dtype=<class 'numpy.int8'>
  idx=  6 | name=model/batch_normalization_1/FusedBatchNo | shape=[30] | dtype=<class 'numpy.int32'>
  idx=  7 | name=model/dense/MatMul                       | shape=[  16 1920] | dtype=<class 'numpy.int8'>
  idx=  8 | name=model/dense/BiasAdd/ReadVariableOp/resou | shape=[16] | dtype=<class 'numpy.int32'>
  idx=  9 | name=model/batch_normalization_2/batchnorm/mu

In [5]:
# @title
# 5) Inspect operators (layers) and identify Conv layers, their weights & biases
# Note: _get_ops_details() is an undocumented but commonly used helper that returns operator list.
ops = interpreter._get_ops_details()

print("\nOperators (ops):")
for i,op in enumerate(ops):
    print(f" op#{i:02d} | name={op['op_name']:20s} | inputs={op['inputs']} | outputs={op['outputs']}")



Operators (ops):
 op#00 | name=RESHAPE              | inputs=[0 1] | outputs=[13]
 op#01 | name=CONV_2D              | inputs=[13  3  4] | outputs=[14]
 op#02 | name=CONV_2D              | inputs=[14  5  6] | outputs=[15]
 op#03 | name=RESHAPE              | inputs=[15  2] | outputs=[16]
 op#04 | name=FULLY_CONNECTED      | inputs=[16  7  8] | outputs=[17]
 op#05 | name=FULLY_CONNECTED      | inputs=[17  9 10] | outputs=[18]
 op#06 | name=FULLY_CONNECTED      | inputs=[18 11 12] | outputs=[19]
 op#07 | name=SOFTMAX              | inputs=[19] | outputs=[20]
 op#08 | name=DELEGATE             | inputs=[ 0  1  2  3  4  5  6  7  8  9 10 11 12] | outputs=[19]


In [6]:
# @title
# 6) For each Conv/Depthwise op, print weight/bias shapes and deduce number of kernels (filters)
conv_ops = []
total_kernels = 0

# Build a quick index => tensor info map
td_map = {t['index']: t for t in tensor_details}

for i,op in enumerate(ops):
    op_name = op['op_name']
    if op_name in ("CONV_2D", "DEPTHWISE_CONV_2D"):
        inputs = op['inputs']
        # common TFLite conv input ordering: [input, filter, bias]  (bias may be optional)
        weight_shape = None
        bias_shape = None
        out_channels = None

        # weight tensor usually at inputs[1]
        if len(inputs) > 1 and inputs[1] in td_map:
            w_idx = inputs[1]
            try:
                w = interpreter.get_tensor(w_idx)
                weight_shape = w.shape
            except Exception:
                weight_shape = td_map[w_idx]['shape']  # fallback to declared shape

        # bias tensor usually at inputs[2] if present
        if len(inputs) > 2 and inputs[2] in td_map:
            b_idx = inputs[2]
            try:
                b = interpreter.get_tensor(b_idx)
                bias_shape = b.shape
            except Exception:
                bias_shape = td_map[b_idx]['shape']

        # Determine number of output filters (robust method):
        if bias_shape is not None and len(bias_shape) >= 1:
            # bias length typically equals output channels (reliable)
            out_channels = int(bias_shape[0])
            reason = "from bias"
        elif weight_shape is not None and len(weight_shape) == 4:
            # Common TF kernel layout: [kH, kW, in_ch, out_ch]
            # Use last dim as out_ch (works for TF -> TFLite typical layout)
            out_channels = int(weight_shape[-1])
            reason = "from weight[-1]"
        elif weight_shape is not None and len(weight_shape) == 1:
            out_channels = int(weight_shape[0])
            reason = "from 1D weight"
        else:
            out_channels = None
            reason = "unknown"

        conv_ops.append({
            "op_index": i,
            "op_name": op_name,
            "inputs": inputs,
            "weight_shape": tuple(weight_shape) if weight_shape is not None else None,
            "bias_shape": tuple(bias_shape) if bias_shape is not None else None,
            "out_channels": out_channels,
            "reason": reason
        })
        if out_channels:
            total_kernels += out_channels

# Print conv summary
print("\nConvolution layers summary:")
for c in conv_ops:
    print(f" op#{c['op_index']:02d} | {c['op_name']:18s} | out_kernels={c['out_channels']:4} | weight={c['weight_shape']} | bias={c['bias_shape']} | ({c['reason']})")

print("\nTotal convolution kernels (sum of out_channels):", total_kernels)



Convolution layers summary:
 op#01 | CONV_2D            | out_kernels=  28 | weight=(28, 10, 4, 1) | bias=(28,) | (from bias)
 op#02 | CONV_2D            | out_kernels=  30 | weight=(30, 10, 4, 28) | bias=(30,) | (from bias)

Total convolution kernels (sum of out_channels): 58


In [7]:
# @title
# 7) Extra: show per-layer parameter counts (weights + biases) when possible
def param_count_for_conv(entry):
    wshape = entry['weight_shape']
    bshape = entry['bias_shape']
    wcount = int(np.prod(wshape)) if wshape is not None else 0
    bcount = int(np.prod(bshape)) if bshape is not None else 0
    return wcount + bcount, wcount, bcount

print("\nPer-conv parameter counts:")
for c in conv_ops:
    total_p, wcount, bcount = param_count_for_conv(c)
    print(f" op#{c['op_index']:02d} | {c['op_name']:18s} | params={total_p:6d} (weights={wcount:6d}, bias={bcount:4d})")



Per-conv parameter counts:
 op#01 | CONV_2D            | params=  1148 (weights=  1120, bias=  28)
 op#02 | CONV_2D            | params= 33630 (weights= 33600, bias=  30)


In [8]:
# ==============================================================================
# 1. TFLite Exact Hardware Math Emulation
# ==============================================================================

def rounding_divide_by_pot(x, exponent):
    """Replicates TFLite's gemmlowp RoundingDivideByPOT for bit-perfect matching."""
    x = np.int64(x) # Prevent NumPy overflow
    mask = (1 << exponent) - 1
    remainder = x & mask
    threshold = (mask >> 1) + (1 if x < 0 else 0)
    return (x >> exponent) + (1 if remainder > threshold else 0)

def multiply_by_quantized_multiplier(x, quantized_multiplier, shift):
    """Replicates the 64-bit to 32-bit scaling block found in standard RTL."""
    x = np.int64(x) # Prevent NumPy overflow
    quantized_multiplier = np.int64(quantized_multiplier)

    left_shift = shift if shift > 0 else 0
    right_shift = -shift if shift < 0 else 0

    # 1. Left shift (if required)
    x_shifted = x * (1 << left_shift)

    # 2. 64-bit Multiplication
    total = x_shifted * quantized_multiplier

    # 3. Take upper 32 bits with specific rounding
    val = total + (1 << 30)
    val = val >> 31 # Simulates C++ 2's complement right shift

    # 4. Right shift with rounding
    if right_shift > 0:
        val = rounding_divide_by_pot(val, right_shift)

    return val

def quantize_multiplier(real_multiplier):
    """C++ std::round emulator for bit-perfect hardware multipliers."""
    if real_multiplier == 0.0:
        return 0, 0
    mantissa, exponent = math.frexp(real_multiplier)
    q_fixed = math.floor(mantissa * (1 << 31) + 0.5)
    if q_fixed == (1 << 31):
        q_fixed = int(q_fixed / 2)
        exponent += 1
    return int(q_fixed), exponent


# ==============================================================================
# 2. Load Model & Generate "Golden" Output
# ==============================================================================

# Initialize interpreter
interpreter = tf.lite.Interpreter(
    model_path="cnn_s_quantized.tflite",
    experimental_preserve_all_tensors=True
)
interpreter.allocate_tensors()

# Load user input and force it to int8
x = np.load('in0.npy')
x = x.astype(np.int8)

# Feed the interpreter and invoke to populate intermediate tensors
interpreter.set_tensor(0, x)
interpreter.invoke()

# Extract the execution parameters
tensor_details = {t['index']: t for t in interpreter.get_tensor_details()}

input_data = interpreter.get_tensor(13)     # Shape: [1, 49, 10, 1]
weights = interpreter.get_tensor(3)         # Shape: [28, 10, 4, 1]
biases = interpreter.get_tensor(4)          # Shape: [28]
golden_output = interpreter.get_tensor(14)  # Shape: [1, 40, 7, 28]

# Extract Quantization Parameters (Cast to int to strip numpy wrappers)
input_zp = np.int64(tensor_details[13]['quantization_parameters']['zero_points'][0])
output_zp = np.int64(tensor_details[14]['quantization_parameters']['zero_points'][0])

input_scale = tensor_details[13]['quantization_parameters']['scales'][0]
output_scale = tensor_details[14]['quantization_parameters']['scales'][0]
weight_scales = tensor_details[3]['quantization_parameters']['scales']

# Calculate multiplier and shift arrays for the 28 channels
multipliers = []
shifts = []
for w_scale in weight_scales:
    real_multiplier = (input_scale * w_scale) / output_scale
    m_quant, shift = quantize_multiplier(real_multiplier)
    multipliers.append(m_quant)
    shifts.append(shift)


# ==============================================================================
# 3. Manual Pure-Loop Implementation (Hardware Reference)
# ==============================================================================

out_batch, out_h_dim, out_w_dim, out_c_dim = golden_output.shape
_, filter_h, filter_w, _ = weights.shape

# Prepare empty array for our manual calculations
manual_output = np.zeros_like(golden_output, dtype=np.int8)

# Loop 1: Output Channels (to avoid redundant multiplier fetching)
for out_c in range(out_c_dim):
    # Fetch parameters specific to this filter
    bias_val = np.int64(biases[out_c])
    m_quant = multipliers[out_c]
    shift = shifts[out_c]

    # Loop 2 & 3: Spatial sliding window
    for out_h in range(out_h_dim):
        for out_w in range(out_w_dim):
            acc = 0

            # Loop 4 & 5: The 2D MAC unit (Stride 1, VALID padding)
            for f_h in range(filter_h):
                for f_w in range(filter_w):

                    # VERY IMPORTANT: Cast numpy int8 to Python 'int' to prevent
                    # 8-bit overflow during the multiply-accumulate process!
                    in_val = np.int64(input_data[0, out_h + f_h, out_w + f_w, 0])
                    w_val = np.int64(weights[out_c, f_h, f_w, 0])

                    acc += (in_val - input_zp) * w_val

            # Hardware Pipeline: Bias -> Scale -> ZeroPoint -> Activation
            acc += bias_val
            acc = multiply_by_quantized_multiplier(acc, m_quant, shift)
            acc += output_zp

            # Activation Function: Fused ReLU bounded to int8
            # Because it's a ReLU, the lower limit is the mathematical 0.0,
            # which in int8 is exactly represented by the `output_zp`.
            acc = max(output_zp, min(127, acc))

            # Write out exactly 1 pixel
            manual_output[0, out_h, out_w, out_c] = acc


# ==============================================================================
# 4. Verification
# ==============================================================================

difference = np.sum(manual_output != golden_output)
print(f"Total mismatched values: {difference} out of {manual_output.size}")

if difference == 0:
    print("\nSUCCESS! The manual integer loops perfectly matched the TFLite output.")
else:
    print("\nMismatch detected. Here is a sample of the differences:")
    mask = manual_output != golden_output
    print("Manual Output :", manual_output[mask][:10])
    print("Golden Output :", golden_output[mask][:10])

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:465: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


Total mismatched values: 10 out of 7840

Mismatch detected. Here is a sample of the differences:
Manual Output : [  46   89 -120  -22   56  -96  -98  -49  -77 -113]
Golden Output : [  45   88 -121  -23   55  -97  -99  -50  -78 -114]


In [9]:
# ==============================================================================
# Layer 2: Parameter Extraction
# ==============================================================================

# Input from Layer 1 is Tensor 14. Output of Layer 2 is Tensor 15.
l2_input = golden_output                      # Shape: [1, 40, 7, 28]
l2_weights = interpreter.get_tensor(5)        # Shape: [30, 10, 4, 28]
l2_biases = interpreter.get_tensor(6)         # Shape: [30]
l2_golden_output = interpreter.get_tensor(15) # Shape: [1, 16, 4, 30]

# Extract Zero Points
l2_input_zp = int(tensor_details[14]['quantization_parameters']['zero_points'][0])
l2_output_zp = int(tensor_details[15]['quantization_parameters']['zero_points'][0])

# Extract Scales
l2_input_scale = tensor_details[14]['quantization_parameters']['scales'][0]
l2_output_scale = tensor_details[15]['quantization_parameters']['scales'][0]
l2_weight_scales = tensor_details[5]['quantization_parameters']['scales']

# Calculate multipliers and shifts for all 30 output channels
l2_multipliers = []
l2_shifts = []
for w_scale in l2_weight_scales:
    real_multiplier = (l2_input_scale * w_scale) / l2_output_scale
    m_quant, shift = quantize_multiplier(real_multiplier)
    l2_multipliers.append(m_quant)
    l2_shifts.append(shift)


# ==============================================================================
# Layer 2: Conv2D with Stride (2, 1) and VALID Padding
# ==============================================================================

out_batch, out_h_dim, out_w_dim, out_c_dim = l2_golden_output.shape
_, filter_h, filter_w, in_c_dim = l2_weights.shape

# The Stride Parameters
stride_h = 2
stride_w = 1

l2_manual_output = np.zeros_like(l2_golden_output, dtype=np.int8)

# 1. Output Channels (The 30 Filters)
for out_c in range(out_c_dim):
    bias_val = int(l2_biases[out_c])
    m_quant = l2_multipliers[out_c]
    shift = l2_shifts[out_c]

    # 2 & 3. Spatial sliding window over the OUTPUT dimensions
    for out_h in range(out_h_dim):
        for out_w in range(out_w_dim):
            acc = 0

            # Map the output coordinate to the starting input coordinate using the stride
            in_h_start = out_h * stride_h
            in_w_start = out_w * stride_w

            # 4 & 5. MAC Loop over the Filter spatial dimensions
            for f_h in range(filter_h):
                for f_w in range(filter_w):

                    # 6. Accumulate across all Input Channels (Depth)
                    for in_c in range(in_c_dim):

                        in_val = int(l2_input[0, in_h_start + f_h, in_w_start + f_w, in_c])
                        w_val = int(l2_weights[out_c, f_h, f_w, in_c])

                        acc += (in_val - l2_input_zp) * w_val

            # Hardware Pipeline
            acc += bias_val
            acc = multiply_by_quantized_multiplier(acc, m_quant, shift)
            acc += l2_output_zp

            # Fused ReLU bounded to int8
            acc = max(l2_output_zp, min(127, acc))

            l2_manual_output[0, out_h, out_w, out_c] = acc

# Verification
l2_diff = np.sum(l2_manual_output != l2_golden_output)
print(f"Layer 2 mismatched values: {l2_diff} out of {l2_manual_output.size}")

Layer 2 mismatched values: 0 out of 1920


In [10]:
# ==============================================================================
# Layer 3: Dense Layer (Fully Connected) - Op#04
# ==============================================================================

# 1. Flatten the output of Layer 2 (Zero-cost in hardware)
# Note: Tensor 16 is literally just Tensor 15 viewed as 1D.
l3_input = l2_golden_output.flatten()         # Shape: [1920]

l3_weights = interpreter.get_tensor(7)        # Shape: [16, 1920]
l3_biases = interpreter.get_tensor(8)         # Shape: [16]
l3_golden_output = interpreter.get_tensor(17) # Shape: [1, 16]

# 2. Extract Quantization Parameters
l3_input_zp = int(tensor_details[16]['quantization_parameters']['zero_points'][0])
l3_output_zp = int(tensor_details[17]['quantization_parameters']['zero_points'][0])

l3_input_scale = tensor_details[16]['quantization_parameters']['scales'][0]
l3_output_scale = tensor_details[17]['quantization_parameters']['scales'][0]
l3_weight_scales = tensor_details[7]['quantization_parameters']['scales']

# 3. Calculate per-channel multipliers (16 output neurons = 16 multipliers)
l3_multipliers = []
l3_shifts = []
for w_scale in l3_weight_scales:
    real_multiplier = (l3_input_scale * w_scale) / l3_output_scale
    m_quant, shift = quantize_multiplier(real_multiplier)
    l3_multipliers.append(m_quant)
    l3_shifts.append(shift)

# ==============================================================================
# Execution Loops (Corrected for Per-Tensor Quantization)
# ==============================================================================

out_neurons = l3_weights.shape[0] # 16
in_features = l3_weights.shape[1] # 1920

l3_manual_output = np.zeros_like(l3_golden_output, dtype=np.int8)

# Loop 1: Iterate over the 16 output neurons
for out_n in range(out_neurons):
    bias_val = int(l3_biases[out_n])

    # Safely handle Per-Tensor (1 scale) vs Per-Channel (N scales)
    scale_idx = out_n if len(l3_multipliers) > 1 else 0
    m_quant = l3_multipliers[scale_idx]
    shift = l3_shifts[scale_idx]

    acc = 0

    # Loop 2: The 1D Dot Product (Multiply-Accumulate across all 1920 features)
    for in_f in range(in_features):
        in_val = int(l3_input[in_f])
        w_val = int(l3_weights[out_n, in_f])

        acc += (in_val - l3_input_zp) * w_val

    # Hardware Pipeline
    acc += bias_val
    acc = multiply_by_quantized_multiplier(acc, m_quant, shift)
    acc += l3_output_zp

    # Standard int8 Saturation (No ReLU!)
    acc = max(-128, min(127, acc))

    l3_manual_output[0, out_n] = acc

# Verification
l3_diff = np.sum(l3_manual_output != l3_golden_output)
print(f"Layer 3 mismatched values: {l3_diff} out of {l3_manual_output.size}")

Layer 3 mismatched values: 0 out of 16


In [11]:
# ==============================================================================
# Layer 4: Dense Layer 2 (Fully Connected) - Op#05
# ==============================================================================

# Input is the output of Layer 3
l4_input = l3_golden_output.flatten()         # Shape: [16]

l4_weights = interpreter.get_tensor(9)        # Shape: [128, 16]
l4_biases = interpreter.get_tensor(10)        # Shape: [128]
l4_golden_output = interpreter.get_tensor(18) # Shape: [1, 128]

# Extract Quantization Parameters
l4_input_zp = int(tensor_details[17]['quantization_parameters']['zero_points'][0])
l4_output_zp = int(tensor_details[18]['quantization_parameters']['zero_points'][0])

l4_input_scale = tensor_details[17]['quantization_parameters']['scales'][0]
l4_output_scale = tensor_details[18]['quantization_parameters']['scales'][0]
l4_weight_scales = tensor_details[9]['quantization_parameters']['scales']

# Calculate multipliers and shifts
l4_multipliers = []
l4_shifts = []
for w_scale in l4_weight_scales:
    real_multiplier = (l4_input_scale * w_scale) / l4_output_scale
    m_quant, shift = quantize_multiplier(real_multiplier)
    l4_multipliers.append(m_quant)
    l4_shifts.append(shift)


# ==============================================================================
# Execution Loops
# ==============================================================================

out_neurons_l4 = l4_weights.shape[0] # 128
in_features_l4 = l4_weights.shape[1] # 16

l4_manual_output = np.zeros_like(l4_golden_output, dtype=np.int8)

# Loop 1: Iterate over the 128 output neurons
for out_n in range(out_neurons_l4):
    bias_val = int(l4_biases[out_n])

    # Safely handle Per-Tensor vs Per-Channel Quantization
    scale_idx = out_n if len(l4_multipliers) > 1 else 0
    m_quant = l4_multipliers[scale_idx]
    shift = l4_shifts[scale_idx]

    acc = 0

    # Loop 2: 1D Dot Product across the 16 input features
    for in_f in range(in_features_l4):
        in_val = int(l4_input[in_f])
        w_val = int(l4_weights[out_n, in_f])

        acc += (in_val - l4_input_zp) * w_val

    # Hardware Pipeline
    acc += bias_val
    acc = multiply_by_quantized_multiplier(acc, m_quant, shift)
    acc += l4_output_zp

    # Standard int8 Saturation (No ReLU)
    acc = max(-128, min(127, acc))

    l4_manual_output[0, out_n] = acc

# Verification
l4_diff = np.sum(l4_manual_output != l4_golden_output)
print(f"Layer 4 mismatched values: {l4_diff} out of {l4_manual_output.size}")

Layer 4 mismatched values: 0 out of 128


In [12]:
# ==============================================================================
# Layer 5: Final Dense Layer (Classification) - Op#06
# ==============================================================================

# Input is the output of Layer 4
l5_input = l4_golden_output.flatten()         # Shape: [128]

l5_weights = interpreter.get_tensor(11)       # Shape: [12, 128]
l5_biases = interpreter.get_tensor(12)        # Shape: [12]
l5_golden_output = interpreter.get_tensor(19) # Shape: [1, 12]

# Extract Quantization Parameters
l5_input_zp = int(tensor_details[18]['quantization_parameters']['zero_points'][0])
l5_output_zp = int(tensor_details[19]['quantization_parameters']['zero_points'][0])

l5_input_scale = tensor_details[18]['quantization_parameters']['scales'][0]
l5_output_scale = tensor_details[19]['quantization_parameters']['scales'][0]
l5_weight_scales = tensor_details[11]['quantization_parameters']['scales']

# Calculate multipliers and shifts
l5_multipliers = []
l5_shifts = []
for w_scale in l5_weight_scales:
    real_multiplier = (l5_input_scale * w_scale) / l5_output_scale
    m_quant, shift = quantize_multiplier(real_multiplier)
    l5_multipliers.append(m_quant)
    l5_shifts.append(shift)


# ==============================================================================
# Execution Loops
# ==============================================================================

out_neurons_l5 = l5_weights.shape[0] # 12
in_features_l5 = l5_weights.shape[1] # 128

l5_manual_output = np.zeros_like(l5_golden_output, dtype=np.int8)

# Loop 1: Iterate over the 12 output classes
for out_n in range(out_neurons_l5):
    bias_val = int(l5_biases[out_n])

    # Safely handle Per-Tensor vs Per-Channel Quantization
    scale_idx = out_n if len(l5_multipliers) > 1 else 0
    m_quant = l5_multipliers[scale_idx]
    shift = l5_shifts[scale_idx]

    acc = 0

    # Loop 2: 1D Dot Product across the 128 input features
    for in_f in range(in_features_l5):
        in_val = int(l5_input[in_f])
        w_val = int(l5_weights[out_n, in_f])

        acc += (in_val - l5_input_zp) * w_val

    # Hardware Pipeline
    acc += bias_val
    acc = multiply_by_quantized_multiplier(acc, m_quant, shift)
    acc += l5_output_zp

    # Standard int8 Saturation (No ReLU)
    acc = max(-128, min(127, acc))

    l5_manual_output[0, out_n] = acc

# Verification
l5_diff = np.sum(l5_manual_output != l5_golden_output)
print(f"Layer 5 mismatched values: {l5_diff} out of {l5_manual_output.size}")

print("\nFinal Output Logits (Pre-Softmax):")
print("Manual :", l5_manual_output[0])
print("Golden :", l5_golden_output[0])
print(f"Predicted Class Index: {np.argmax(l5_manual_output[0])}")

Layer 5 mismatched values: 0 out of 12

Final Output Logits (Pre-Softmax):
Manual : [-128   63    1   41  -17    7   31   34    3  -10    2   69]
Golden : [-128   63    1   41  -17    7   31   34    3  -10    2   69]
Predicted Class Index: 11


In [13]:
def numpy_to_c_array(arr):
    """Recursively formats a NumPy array into a nested C-style string."""
    if arr.ndim == 1:
        return "{" + ", ".join(map(str, arr)) + "}"
    else:
        return "{" + ", ".join(numpy_to_c_array(sub) for sub in arr) + "}"

def get_c_dims(shape):
    """Converts a Python shape tuple into C array dimension brackets."""
    return "".join(f"[{d}]" for d in shape)

# ==============================================================================
# 1. Initialize Interpreter
# ==============================================================================
interpreter = tf.lite.Interpreter(model_path="cnn_s_quantized.tflite")
interpreter.allocate_tensors()
tensor_details = {t['index']: t for t in interpreter.get_tensor_details()}

# We map out the exact tensor indices for our 5 execution layers
layers = [
    {"name": "conv1", "in_idx": 13, "w_idx": 3, "b_idx": 4, "out_idx": 14},
    {"name": "conv2", "in_idx": 14, "w_idx": 5, "b_idx": 6, "out_idx": 15},
    {"name": "dense1", "in_idx": 16, "w_idx": 7, "b_idx": 8, "out_idx": 17},
    {"name": "dense2", "in_idx": 17, "w_idx": 9, "b_idx": 10, "out_idx": 18},
    {"name": "dense3", "in_idx": 18, "w_idx": 11, "b_idx": 12, "out_idx": 19},
]

# ==============================================================================
# 2. Extract and Dump to C Header File
# ==============================================================================
with open("model_parameters.h", "w") as f:
    f.write("#ifndef MODEL_PARAMETERS_H\n")
    f.write("#define MODEL_PARAMETERS_H\n\n")
    f.write("#include <stdint.h>\n\n")

    for layer in layers:
        name = layer["name"]
        print(f"Extracting parameters for {name}...")

        # 1. Get Static Tensors
        weights = interpreter.get_tensor(layer["w_idx"])
        biases = interpreter.get_tensor(layer["b_idx"])

        # 2. Get Quantization Params
        in_zp = int(tensor_details[layer["in_idx"]]['quantization_parameters']['zero_points'][0])
        out_zp = int(tensor_details[layer["out_idx"]]['quantization_parameters']['zero_points'][0])

        in_scale = tensor_details[layer["in_idx"]]['quantization_parameters']['scales'][0]
        out_scale = tensor_details[layer["out_idx"]]['quantization_parameters']['scales'][0]
        w_scales = tensor_details[layer["w_idx"]]['quantization_parameters']['scales']

        # 3. Calculate Multipliers and Shifts
        multipliers = []
        shifts = []
        for w_scale in w_scales:
            real_multiplier = (in_scale * w_scale) / out_scale
            m_quant, shift = quantize_multiplier(real_multiplier)
            multipliers.append(m_quant)
            shifts.append(shift)

        multipliers = np.array(multipliers, dtype=np.int32)
        shifts = np.array(shifts, dtype=np.int32)

        # ==========================================================================
        # 4. Write C Declarations
        # ==========================================================================
        f.write(f"// ==================================================\n")
        f.write(f"// Layer: {name.upper()}\n")
        f.write(f"// ==================================================\n")

        # Zero Points
        f.write(f"const int32_t {name}_input_zp = {in_zp};\n")
        f.write(f"const int32_t {name}_output_zp = {out_zp};\n\n")

        # Weights (Multidimensional)
        w_dims = get_c_dims(weights.shape)
        w_str = numpy_to_c_array(weights)
        f.write(f"const int8_t {name}_weights{w_dims} = {w_str};\n\n")

        # Biases (1D)
        b_dims = get_c_dims(biases.shape)
        b_str = numpy_to_c_array(biases)
        f.write(f"const int32_t {name}_biases{b_dims} = {b_str};\n\n")

        # Multipliers (1D)
        m_dims = get_c_dims(multipliers.shape)
        m_str = numpy_to_c_array(multipliers)
        f.write(f"const int32_t {name}_multipliers{m_dims} = {m_str};\n\n")

        # Shifts (1D)
        s_dims = get_c_dims(shifts.shape)
        s_str = numpy_to_c_array(shifts)
        f.write(f"const int32_t {name}_shifts{s_dims} = {s_str};\n\n")

    f.write("#endif // MODEL_PARAMETERS_H\n")

print("\nSuccess! All parameters have been safely extracted to 'model_parameters.h'")

Extracting parameters for conv1...
Extracting parameters for conv2...
Extracting parameters for dense1...
Extracting parameters for dense2...
Extracting parameters for dense3...

Success! All parameters have been safely extracted to 'model_parameters.h'


In [14]:
interpreter = tf.lite.Interpreter(
    model_path="cnn_s_quantized.tflite",
    experimental_preserve_all_tensors=True
)
interpreter.allocate_tensors()
interpreter.set_tensor(0, x)
interpreter.invoke()
tensor_details = {t['index']: t for t in interpreter.get_tensor_details()}

(f"const int8_t conv1_input[1][49][10][1] = {numpy_to_c_array(interpreter.get_tensor(13))};\n\n")

'const int8_t conv1_input[1][49][10][1] = {{{{-88}, {-70}, {98}, {18}, {-51}, {-93}, {-53}, {71}, {-124}, {-26}}, {{-68}, {62}, {96}, {54}, {-68}, {-82}, {39}, {25}, {32}, {27}}, {{126}, {-32}, {-128}, {-119}, {-50}, {62}, {-64}, {-46}, {29}, {71}}, {{107}, {61}, {-79}, {-113}, {21}, {107}, {-55}, {-43}, {-8}, {-106}}, {{-12}, {42}, {-55}, {96}, {79}, {29}, {27}, {-112}, {-23}, {-69}}, {{12}, {-68}, {82}, {-54}, {118}, {108}, {-119}, {93}, {-95}, {-40}}, {{70}, {33}, {41}, {-104}, {54}, {17}, {103}, {77}, {71}, {-91}}, {{-104}, {81}, {-47}, {-26}, {-90}, {-49}, {25}, {-35}, {-124}, {-37}}, {{86}, {98}, {125}, {84}, {-84}, {-9}, {-5}, {-91}, {-86}, {-128}}, {{-109}, {-110}, {-111}, {1}, {-52}, {-82}, {-38}, {-34}, {117}, {115}}, {{-2}, {57}, {126}, {-89}, {95}, {-23}, {-94}, {115}, {84}, {80}}, {{44}, {-87}, {-78}, {-12}, {-49}, {11}, {-88}, {-85}, {7}, {-20}}, {{-36}, {103}, {-59}, {12}, {12}, {72}, {64}, {22}, {34}, {111}}, {{30}, {126}, {34}, {-88}, {62}, {110}, {-68}, {-105}, {-76},

In [15]:
# ==============================================================================
# 1. Initialize Interpreter and Run Inference
# ==============================================================================
# Force preservation of intermediate tensors and disable XNNPACK for bit-perfect reference
interpreter = tf.lite.Interpreter(
    model_path="cnn_s_quantized.tflite",
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
)
interpreter.allocate_tensors()

# Load user input
try:
    x = np.load('in0.npy').astype(np.int8)
except FileNotFoundError:
    print("Error: 'in0.npy' not found. Please ensure the file is in the same directory.")
    exit(1)

interpreter.set_tensor(0, x)
interpreter.invoke()

# ==============================================================================
# 2. Extract Golden Activations
# ==============================================================================
# Map the execution layers to their respective output tensor indices
layers = [
    {"name": "conv1",  "out_idx": 14}, # Shape: [1, 40, 7, 28]
    {"name": "conv2",  "out_idx": 15}, # Shape: [1, 16, 4, 30]
    {"name": "dense1", "out_idx": 17}, # Shape: [1, 16]
    {"name": "dense2", "out_idx": 18}, # Shape: [1, 128]
    {"name": "dense3", "out_idx": 19}, # Shape: [1, 12]
]

with open("golden_activations.h", "w") as f:
    f.write("#ifndef GOLDEN_ACTIVATIONS_H\n")
    f.write("#define GOLDEN_ACTIVATIONS_H\n\n")
    f.write("#include <stdint.h>\n\n")

    for layer in layers:
        name = layer["name"]
        out_idx = layer["out_idx"]

        print(f"Extracting golden activations for {name} (Tensor {out_idx})...")

        # Get the populated intermediate tensor
        golden_tensor = interpreter.get_tensor(out_idx)

        # Format for C
        dims = get_c_dims(golden_tensor.shape)
        arr_str = numpy_to_c_array(golden_tensor)

        f.write(f"// ==================================================\n")
        f.write(f"// Layer: {name.upper()} (Tensor {out_idx})\n")
        f.write(f"// Shape: {list(golden_tensor.shape)}\n")
        f.write(f"// ==================================================\n")
        f.write(f"const int8_t {name}_golden_output{dims} = {arr_str};\n\n")

    f.write("#endif // GOLDEN_ACTIVATIONS_H\n")

print("\nSuccess! Golden activations exported to 'golden_activations.h'")

Extracting golden activations for conv1 (Tensor 14)...
Extracting golden activations for conv2 (Tensor 15)...
Extracting golden activations for dense1 (Tensor 17)...
Extracting golden activations for dense2 (Tensor 18)...
Extracting golden activations for dense3 (Tensor 19)...

Success! Golden activations exported to 'golden_activations.h'
